# Ingest Customers Dataset file

1. read the file using spark dataframe reader API
- Define the Schema 
2. Add metadata columns
- source file 
- ingestion timestamp
3. write to the bronze delta table

In [0]:
%run ../01-common/01.bronze_helper

In [0]:
#Imports
from pyspark.sql.functions import col
from pyspark.sql.functions import current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, IntegerType,TimestampType

In [0]:
source_file="/Volumes/olist_catalog/landing/files/olist_customers_dataset.csv"
table_name = "olist_catalog.bronze.customers"

In [0]:
customers_schema = StructType([
    StructField("customer_id",StringType()),
    StructField("customer_unique_id",StringType()),
    StructField("customer_zip_code_prefix",IntegerType()),
    StructField("customer_city",StringType()),
    StructField("customer_state",StringType())
])

In [0]:
customers_df = (
    spark.read
    .format("csv")
    .option("header","True")
    .schema(customers_schema)
    .load(source_file)
    )

In [0]:
customers_df_final=add_ingestion_data(customers_df)

In [0]:
display(customers_df_final)

In [0]:
(
    customers_df_final.write
    .format('delta')
    .mode('overwrite')
    .saveAsTable(table_name)
)

In [0]:
%sql
select * from olist_catalog.bronze.customers